# 1.06 线性回归与梯度下降：所有训练循环的骨架

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.06_linear_regression.ipynb)


## 为什么要学

线性回归是最简单的模型，但它的训练流程（**数据 → 模型 → 损失 → 梯度 → 更新**）和训练 π0 一模一样。区别只是模型从 `y = wx + b` 变成了 30 亿参数的 Transformer。

本课用三种方式实现线性回归：

1. **纯手写**：自己推梯度公式；
2. **autograd**：让 PyTorch 求梯度；
3. **`nn.Module` + `optim`**：工业界的标准写法。

参考：[d2l 3.2 线性回归的从零开始实现](https://zh.d2l.ai/chapter_linear-networks/linear-regression-scratch.html)。

---

## 1. 问题设定

给定数据 $\{(\mathbf{x}_i, y_i)\}_{i=1}^N$，$\mathbf{x}_i\in\mathbb{R}^d$，假设

$$
\hat y = \mathbf{w}^\top\mathbf{x} + b
$$

损失函数用均方误差（上一课讲过：这等价于高斯噪声下的最大似然）：

$$
L(\mathbf{w}, b) = \frac{1}{N}\sum_{i=1}^N\left(\mathbf{w}^\top\mathbf{x}_i + b - y_i\right)^2
$$

### 梯度推导

记残差 $e_i = \mathbf{w}^\top\mathbf{x}_i + b - y_i$，链式法则：

$$
\frac{\partial L}{\partial \mathbf{w}} = \frac{2}{N}\sum_i e_i\,\mathbf{x}_i = \frac{2}{N}X^\top\mathbf{e},\qquad
\frac{\partial L}{\partial b} = \frac{2}{N}\sum_i e_i
$$

### 闭式解（正规方程）

令梯度为 0（把 $b$ 并入 $\mathbf{w}$，在 $X$ 后面补一列 1），得到 $X^\top X\mathbf{w} = X^\top\mathbf{y}$。线性回归有闭式解，但神经网络没有，所以我们需要梯度下降。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
true_w = torch.tensor([2.0, -3.4]); true_b = 4.2
N = 1000
X = torch.randn(N, 2)
y = X @ true_w + true_b + 0.1 * torch.randn(N)     # 加一点噪声

# 闭式解：在 X 后补一列 1 来表示偏置 b
X1 = torch.cat([X, torch.ones(N, 1)], dim=1)
w_closed = torch.linalg.solve(X1.T @ X1, X1.T @ y)
print("闭式解 w, b:", w_closed)

---

## 2. 方式一：纯手写梯度 + 小批量随机梯度下降（SGD）

每次只用一小批（mini-batch）数据估计梯度，而不是全部 $N$ 个：

- 计算快（一个 batch 放进 GPU）；
- 梯度带噪声，反而有助于跳出不好的局部极小值。

### 练习 1：小批量数据迭代器

写一个生成器：每个 epoch 先打乱索引，再按 `batch_size` 依次产出 `(X_batch, y_batch)`。最后一批可以不满。

In [ ]:
def data_iter(X, y, batch_size, shuffle=True):
    N = len(X)
    # TODO: idx = torch.randperm(N) 或 torch.arange(N)；然后按 batch_size 切片，用 yield 产出
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def data_iter(X, y, batch_size, shuffle=True):
    N = len(X)
    idx = torch.randperm(N) if shuffle else torch.arange(N)
    for start in range(0, N, batch_size):
        b = idx[start : start + batch_size]
        yield X[b], y[b]

In [ ]:
# ✅ 自测
batches = list(data_iter(X, y, 300))
assert len(batches) == 4 and batches[-1][0].shape == (100, 2)
assert sum(len(b[0]) for b in batches) == N
print("通过 ✅")

### 练习 2：手写梯度

按照上面推导的公式，实现 MSE 对 $\mathbf{w}, b$ 的梯度。

In [ ]:
def mse_grad(Xb, yb, w, b):
    """Xb: (B, d), yb: (B,), w: (d,), b: 标量 -> (dw (d,), db 标量)"""
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def mse_grad(Xb, yb, w, b):
    """Xb: (B, d), yb: (B,), w: (d,), b: 标量 -> (dw (d,), db 标量)"""
    B = len(Xb)
    e = Xb @ w + b - yb             # (B,) 残差
    dw = 2.0 / B * (Xb.T @ e)       # (d,)
    db = 2.0 / B * e.sum()
    return dw, db

In [ ]:
# ✅ 自测：与 autograd 对比
w_ = torch.randn(2, requires_grad=True); b_ = torch.tensor(0.3, requires_grad=True)
((X[:64] @ w_ + b_ - y[:64]) ** 2).mean().backward()
dw, db = mse_grad(X[:64], y[:64], w_.detach(), b_.detach())
assert torch.allclose(dw, w_.grad, atol=1e-5) and torch.allclose(db, b_.grad, atol=1e-5)
print("通过 ✅")

In [ ]:
# 用手写梯度训练
w = torch.zeros(2); b = torch.tensor(0.0); lr = 0.1
losses = []
for epoch in range(5):
    for Xb, yb in data_iter(X, y, 32):
        dw, db = mse_grad(Xb, yb, w, b)
        w -= lr * dw; b -= lr * db
    losses.append(((X @ w + b - y) ** 2).mean().item())
    print(f"epoch {epoch}  loss {losses[-1]:.5f}  w={w.numpy().round(3)}  b={b.item():.3f}")

---

## 3. 方式二：用 autograd 求梯度

把 `w, b` 设置为 `requires_grad=True`，梯度由 `backward()` 自动计算。更新参数时要放在 `torch.no_grad()` 里，因为"更新"这个操作本身不应该被记录到计算图中。

In [ ]:
w = torch.zeros(2, requires_grad=True); b = torch.zeros(1, requires_grad=True)
for epoch in range(5):
    for Xb, yb in data_iter(X, y, 32):
        loss = ((Xb @ w + b - yb) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= 0.1 * w.grad; b -= 0.1 * b.grad
            w.grad.zero_(); b.grad.zero_()      # 别忘了清零！
print("autograd 版本:", w.detach().numpy().round(3), b.item())

---

## 4. 方式三：`nn.Module` + `optim`：标准写法

这是你在 openpi、FluxVLA、LeRobot 里会看到的写法。**请把下面的"训练五步"刻在脑子里**：

In [ ]:
import torch.nn as nn

model = nn.Linear(2, 1)                          # 模型：自动创建 weight (1,2) 与 bias (1,)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for epoch in range(5):
    for Xb, yb in data_iter(X, y, 32):
        pred = model(Xb).squeeze(-1)     # ① 前向：(B,1) -> (B,)
        loss = loss_fn(pred, yb)         # ② 计算损失
        optimizer.zero_grad()            # ③ 梯度清零
        loss.backward()                  # ④ 反向传播
        optimizer.step()                 # ⑤ 更新参数
print("nn.Module 版本:", model.weight.data, model.bias.data)

> **常见 bug 清单**（遇到 loss 不降先查这些）：
> 1. 忘了 `zero_grad()`：梯度不断累加，等效学习率越来越大；
> 2. `pred` 形状 `(B,1)` 和 `yb` 形状 `(B,)` 直接相减：广播成 `(B,B)`，loss 完全错了，**但不报错**！（`nn.MSELoss` 会给警告）
> 3. 学习率太大导致发散，或太小几乎不动。

### 练习 3：亲手复现"形状广播 bug"

计算 `pred (B,1)` 和 `y (B,)` 的 MSE，分别用"错误写法"和"正确写法"，看看差别有多大。

In [ ]:
def buggy_and_correct_mse(pred, target):
    """pred: (B, 1), target: (B,) -> (错误的 mse, 正确的 mse)"""
    # TODO: 错误写法：((pred - target) ** 2).mean()；正确写法：先把 pred squeeze 成 (B,)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def buggy_and_correct_mse(pred, target):
    """pred: (B, 1), target: (B,) -> (错误的 mse, 正确的 mse)"""
    wrong = ((pred - target) ** 2).mean()            # (B,1)-(B,) 广播成 (B,B)！
    right = ((pred.squeeze(-1) - target) ** 2).mean()
    return wrong, right

In [ ]:
# ✅ 自测
pred = model(X[:100]).detach(); target = y[:100]
wrong, right = buggy_and_correct_mse(pred, target)
print(f"错误写法 {wrong:.3f}   正确写法 {right:.5f}")
assert right < 0.05 and wrong > 1.0
print("通过 ✅ —— 养成习惯：计算 loss 前 assert pred.shape == target.shape")

---

## 5. 学习率对训练的影响

In [ ]:
plt.figure(figsize=(5, 3))
for lr in [0.001, 0.01, 0.1, 0.5]:
    m = nn.Linear(2, 1); opt = torch.optim.SGD(m.parameters(), lr=lr); hist = []
    for epoch in range(3):
        for Xb, yb in data_iter(X, y, 32):
            loss = ((m(Xb).squeeze(-1) - yb) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); hist.append(loss.item())
    plt.plot(hist, label=f"lr={lr}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("effect of learning rate"); plt.show()

---

## 小结

- 训练五步：**前向 → 损失 → 清零 → 反向 → 更新**。所有模型都一样。
- 小批量 SGD 用一小批样本估计梯度；每个 epoch 打乱数据。
- 计算损失前**检查形状**，广播 bug 不会报错但会毁掉训练。

**思考题**：如果把特征 `X[:, 0]` 放大 100 倍（比如一个关节角用"度"、另一个用"弧度"），用同样的学习率训练会怎样？这和第 1.01 课的"动作归一化"有什么关系？